# Product geometry encoder definitions

Inputs are valid patch tokens $h_i$, original 2D positions $p_i$ and masks; outputs retain the original patch token shape. Both online and EMA encoders replace ViT within the same JEPA loss and predictor. Import only the tagged `ahr-export` cell; it does not start training.

The old learned kernel is preserved exactly. The normalization control mixes normalized bands:
$$A_{ij}=\sum_r\pi_r\frac{k_r(i,j)}{\sum_{t\in V}k_r(i,t)}.$$
The product geometry kernel factors positions and bounded six-dimensional learned semantic embeddings:
$$k_r(i,j)=\exp[-\|p_i-p_j\|^2/(2\ell_{p,r}^2)-\|z_i^q-z_j^k\|^2/(2\ell_{c,r}^2)].$$
Positive global per-head lengths start equal and stay in $[.03,2]$. Normalization/product operators are exactly equal at common initialization. The spherical model normalizes nonzero linear feature/position maps to $S^5$ and uses $\exp[-(1-u_i^q\cdot u_j^k)/\tau_r]$, $\tau_r=\ell_{c,r}^2$. Exact zero maps return the first-axis unit vector and zero derivative on that exceptional branch. No smooth invertible sphere chart is claimed.

Three-view kernels use distinct small embeddings and positive source/receiver memberships:
$$A_{ij}=\sum_v b_{v,i}\sum_r\pi_r\frac{a_{v,j}k_{v,r}(i,j)}{\sum_{t\in V}a_{v,t}k_{v,r}(i,t)}.$$
Original positions persist. First-view maps share baseline initialization; extra maps receive deterministic tiny offsets. Additional active parameters are reported. Views are not proved manifold charts. All kernels are exact dense computations; no FMM speedup is implemented.

Patch stem, residual pointwise MLPs, normalization, EMA target and predictor are inherited from Notebook 11. No encoder receives labels. Imports take seconds. Checkpoints and exact resume are owned by the consumer Notebook 13 runner; this definition module has no mutable run state.

In [ ]:
"""Exact dense product-space smooth positive kernels. No manifold-chart/FMM-speed claim."""
from pathlib import Path
import copy, importlib.util, math, sys
import torch
from torch import nn
from torch.nn import functional as F
from fmm_jepa.bootstrap import TaggedNotebookLoader
from fmm_jepa.models import JEPA, Predictor, patch_coordinates
_ROOT=Path(__file__).resolve().parents[2]
if 'fmm11_encoder' not in sys.modules:
    _path=_ROOT/'experiments/11_manifold_long/encoder.ipynb'
    _spec=importlib.util.spec_from_loader('fmm11_encoder',TaggedNotebookLoader(_path))
    _module=importlib.util.module_from_spec(_spec);sys.modules['fmm11_encoder']=_module
    try:_spec.loader.exec_module(_module)
    except BaseException:sys.modules.pop('fmm11_encoder',None);raise
OLD=sys.modules['fmm11_encoder']
BACKEND='exact-dense-product-geometry-v1'
VARIANTS=('vit','learned_kernel','normalized_kernel','product_kernel','sphere_kernel','multi_view_kernel')

def _squared_distance(query,source):
    return (query.square().sum(-1)[...,:,None]+source.square().sum(-1)[...,None,:]-2*(query@source.transpose(-1,-2))).clamp_min(0)

def exact_unit_vectors(raw):
    """Nonzero vectors normalize exactly; zero vectors use e1 with zero derivative.
    Scale-first norm prevents finite input under/overflow; no epsilon contracts norms.
    """
    maximum=raw.abs().amax(-1,keepdim=True);nonzero=maximum>0
    scaled=raw/torch.where(nonzero,maximum,torch.ones_like(maximum))
    squared=scaled.square().sum(-1,keepdim=True)
    norm=torch.where(nonzero,squared,torch.ones_like(squared)).sqrt()
    unit=scaled/norm;fallback=torch.zeros_like(raw);fallback[...,0]=1
    return torch.where(nonzero,unit,fallback),~nonzero.squeeze(-1)

class GeometryMixer(OLD.NormalizedRBFMixer):
    """Mix separately normalized bands; product lengths separate spatial/content.
    Spherical content exp(-(1-uq·uk)/ell_c²) is a chordal Gaussian, tau=ell_c².
    Three views additionally use positive source and receiver memberships.
    """
    def __init__(self,dim,heads=3,variant='normalized_kernel',coordinate_dim=8,
                 bandwidths=(.15,.35,.75),content_scale=.5,views=3):
        if variant not in VARIANTS[2:]:raise ValueError('Expected a new geometry variant')
        if coordinate_dim!=8 or int(views)!=3:raise ValueError('Protocol fixes2spatial+6semantic dimensions and three views')
        super().__init__(dim,heads,'learned_kernel',coordinate_dim,bandwidths,content_scale)
        self.variant=variant;self.views=3 if variant=='multi_view_kernel' else 1
        if variant in ('product_kernel','sphere_kernel','multi_view_kernel'):
            self.log_content_bandwidths=nn.Parameter(self.log_bandwidths.detach().clone())
        if variant=='multi_view_kernel':
            extra=heads*6
            self.query_extra_content=nn.ModuleList([nn.Linear(dim,extra)for _ in range(2)])
            self.key_extra_content=nn.ModuleList([nn.Linear(dim,extra)for _ in range(2)])
            self.query_extra_position=nn.ModuleList([nn.Linear(2,extra,bias=False)for _ in range(2)])
            self.key_extra_position=nn.ModuleList([nn.Linear(2,extra,bias=False)for _ in range(2)])
            self.receiver_membership=nn.Linear(dim,heads*3);self.source_membership=nn.Linear(dim,heads*3)
            initialize_mixer_from_reference(self,self)
        self.backend='exact dense per-scale/per-view normalized smooth positive product kernels; quadratic token pairs'

    def _coordinates(self,x,indices,valid,grid):
        if self.variant!='sphere_kernel':return super()._coordinates(x,indices,valid,grid)
        dtype=torch.float64 if x.dtype==torch.float64 else torch.float32
        safe=torch.where(valid[...,None],x,torch.zeros_like(x)).to(dtype)
        xy=patch_coordinates(indices.clamp(0,grid[0]*grid[1]-1),grid,dtype=dtype)
        spatial=xy[:,None].expand(-1,self.heads,-1,-1)
        def transform(content,position):
            raw=F.linear(safe,content.weight.to(dtype),content.bias.to(dtype))+F.linear(xy,position.weight.to(dtype))
            raw=raw.reshape(x.shape[0],x.shape[1],self.heads,6).transpose(1,2)
            unit,zero=exact_unit_vectors(raw);return torch.cat((spatial,unit),-1),zero
        query,qzero=transform(self.query_content,self.query_position)
        source,kzero=transform(self.key_content,self.key_position)
        self._last_zero_fallback=(qzero,kzero);return query,source

    def _view_coordinates(self,x,indices,valid,grid):
        first=self._coordinates(x,indices,valid,grid)
        if self.views==1:return [first]
        dtype=first[0].dtype;safe=torch.where(valid[...,None],x,torch.zeros_like(x)).to(dtype)
        xy=first[0][:,0,:,:2];spatial=first[0][...,:2]
        def transform(content,position):
            z=F.linear(safe,content.weight.to(dtype),content.bias.to(dtype))+F.linear(xy,position.weight.to(dtype))
            z=(self.content_scale*z.tanh()).reshape(x.shape[0],x.shape[1],self.heads,6).transpose(1,2)
            return torch.cat((spatial,z),-1)
        return [first]+[(transform(qc,qp),transform(kc,kp))for qc,kc,qp,kp in zip(
            self.query_extra_content,self.key_extra_content,self.query_extra_position,self.key_extra_position)]

    def _band_log_kernels(self,query,source):
        dtype=query.dtype
        ellp=self.log_bandwidths.to(dtype).clamp(math.log(.03),math.log(2.)).exp()
        ellc=(self.log_content_bandwidths if hasattr(self,'log_content_bandwidths') else self.log_bandwidths).to(dtype).clamp(math.log(.03),math.log(2.)).exp()
        dp=_squared_distance(query[...,:2],source[...,:2])
        if self.variant=='sphere_kernel':
            distance=(1-query[...,2:]@source[...,2:].transpose(-1,-2)).clamp_min(0)
            terms=[-dp/(2*ellp[None,:,r,None,None].square())-distance/ellc[None,:,r,None,None].square()for r in range(ellp.shape[-1])]
        else:
            dc=_squared_distance(query[...,2:],source[...,2:])
            # Exact operation parity for normalized/product at equal initial lengths.
            terms=[-.5*(dp/ellp[None,:,r,None,None].square()+dc/ellc[None,:,r,None,None].square())for r in range(ellp.shape[-1])]
        return torch.stack(terms,dim=2)

    def _log_kernel(self,query,source):
        bands=self._band_log_kernels(query,source)
        return torch.logsumexp(bands+self.mixture_logits.to(query.dtype).log_softmax(-1)[None,:,:,None,None],dim=2)

    def _memberships(self,x,dtype):
        def membership(layer):
            logits=F.linear(x.to(dtype),layer.weight.to(dtype),layer.bias.to(dtype));logits=6*(logits/6).tanh()
            return logits.reshape(x.shape[0],x.shape[1],self.heads,3).permute(0,2,3,1).log_softmax(2)
        return membership(self.receiver_membership),membership(self.source_membership)

    def kernel_state(self,x,indices,valid,grid):
        safe=torch.where(valid[...,None],x,torch.zeros_like(x))
        with torch.autocast(device_type=x.device.type,enabled=False):
            coordinates=self._view_coordinates(safe,indices,valid,grid);dtype=coordinates[0][0].dtype
            pi=self.mixture_logits.to(dtype).softmax(-1)[None,:,:,None,None]
            branches=[];band_weights=[];band_logs=[];log_receiver=log_source=None
            if self.views==3:log_receiver,log_source=self._memberships(safe,dtype)
            for view,(query,source)in enumerate(coordinates):
                bands=self._band_log_kernels(query,source);logits=bands
                if log_source is not None:logits=logits+log_source[:,:,view,None,None,:]
                logits=logits.masked_fill(~valid[:,None,None,None,:],float('-inf'))
                per_band=logits.softmax(-1);branch=(pi*per_band).sum(2)
                band_logs.append(bands);band_weights.append(per_band);branches.append(branch)
            weights=branches[0]if self.views==1 else sum(log_receiver[:,:,v,:,None].exp()*branch for v,branch in enumerate(branches))
            weights=torch.where(valid[:,None,:,None],weights,torch.zeros_like(weights))
            state={'query_coordinates':coordinates[0][0],'source_coordinates':coordinates[0][1],
                'query_views':[q for q,k in coordinates],'source_views':[k for q,k in coordinates],
                'log_kernel':self._log_kernel(*coordinates[0]),'weights':weights,
                'log_band_kernels':band_logs,'band_weights':band_weights,'view_weights':branches,
                'geometry':'positions×unit S5 semantic embedding'if self.variant=='sphere_kernel'else'positions×bounded six-dimensional semantic embedding','views':self.views}
            if log_receiver is not None:state.update(receiver_memberships=log_receiver.exp(),source_memberships=log_source.exp())
            if self.variant=='sphere_kernel':state['zero_fallback_masks']=self._last_zero_fallback
            return state

class GeometryEncoderBlock(OLD.KernelEncoderBlock):
    def __init__(self,variant,dim,heads=3,**options):
        nn.Module.__init__(self);self.norm1,self.norm2=nn.LayerNorm(dim),nn.LayerNorm(dim)
        self.mlp=nn.Sequential(nn.Linear(dim,4*dim),nn.GELU(),nn.Linear(4*dim,dim))
        self.mixer=GeometryMixer(dim,heads,variant,**options)

class KernelVisionEncoder(OLD.KernelVisionEncoder):
    def __init__(self,kind='learned_kernel',patch_size=8,dim=192,depth=4,heads=3,
                 coordinate_dim=8,bandwidths=(.15,.35,.75),content_scale=.5,checkpoint=False,views=3):
        if kind=='learned_kernel':
            super().__init__(kind,patch_size,dim,depth,heads,coordinate_dim,bandwidths,content_scale,checkpoint);return
        if kind not in VARIANTS[2:]or min(patch_size,dim,depth)<1:raise ValueError('Unknown encoder/invalid dimensions')
        nn.Module.__init__(self);self.patch_size,self.dim,self.kind=int(patch_size),int(dim),kind
        self.checkpoint_blocks=bool(checkpoint);self.patch_embed=nn.Linear(3*patch_size*patch_size,dim)
        self.blocks=nn.ModuleList([GeometryEncoderBlock(kind,dim,heads=heads,coordinate_dim=coordinate_dim,
            bandwidths=bandwidths,content_scale=content_scale,views=views)for _ in range(depth)])
        self.norm=nn.LayerNorm(dim)

class KernelJEPA(JEPA):
    def __init__(self,config,variant=None):
        nn.Module.__init__(self);self.config=copy.deepcopy(config);options=config.get('kernel13',{})
        variant=variant or options.get('variant','learned_kernel')
        self.context=KernelVisionEncoder(kind=variant,patch_size=config.get('patch_size',8),dim=config.get('dim',192),
            depth=options.get('depth',config.get('depth',4)),heads=options.get('heads',config.get('heads',3)),
            coordinate_dim=options.get('coordinate_dim',8),bandwidths=options.get('bandwidths',(.15,.35,.75)),
            content_scale=options.get('content_scale',.5),checkpoint=options.get('checkpoint',False),views=options.get('views',3))
        self.target=copy.deepcopy(self.context).requires_grad_(False).eval()
        self.predictor=Predictor(self.context.dim,dim=config.get('predictor_dim',128),depth=config.get('predictor_depth',2),heads=config.get('predictor_heads',4))

def build_model(config,kind='semantic',variant=None):
    variant=variant or config.get('kernel13',{}).get('variant','learned_kernel')
    if variant not in VARIANTS:raise ValueError('Explicit registered geometry variant required')
    if kind=='vit'or variant=='vit':
        cfg=copy.deepcopy(config);cfg['kind']='vit';return JEPA(cfg)
    if kind!='semantic':raise ValueError('kind must be vit or semantic')
    return KernelJEPA(config,variant)

def _offset_like(parameter,offset):
    values=torch.arange(parameter.numel(),dtype=torch.float64,device=parameter.device).reshape(parameter.shape)
    return (offset*torch.sin(values+1)).to(parameter.dtype)

@torch.no_grad()
def initialize_mixer_from_reference(destination,reference):
    """Copy individual mixer common maps; no aliasing, deterministic tiny extra views."""
    names=[]
    for name in ('value','projection','query_content','key_content','query_position','key_position'):
        module=getattr(destination,name);module.load_state_dict(getattr(reference,name).state_dict());names.extend(name+'.'+key for key in module.state_dict())
    for name in ('log_bandwidths','mixture_logits'):
        getattr(destination,name).copy_(getattr(reference,name));names.append(name)
    if hasattr(destination,'log_content_bandwidths'):
        destination.log_content_bandwidths.copy_(reference.log_bandwidths);names.append('log_content_bandwidths')
    if getattr(destination,'views',1)==3:
        for view,offset in enumerate((.001,-.001)):
            for prefix in ('query','key'):
                for suffix in ('content','position'):
                    module=getattr(destination,prefix+'_extra_'+suffix)[view];module.load_state_dict(getattr(reference,prefix+'_'+suffix).state_dict())
                    for key,param in module.named_parameters():param.add_(_offset_like(param,offset));names.append(f'{prefix}_extra_{suffix}.{view}.{key}')
        for name,offset in [('receiver_membership',.003),('source_membership',-.003)]:
            module=getattr(destination,name);module.weight.copy_(_offset_like(module.weight,offset));module.bias.zero_();names.extend([name+'.weight',name+'.bias'])
    return names

def parameter_counts(model):
    result=OLD.parameter_counts(model)
    result.update(semantic_views=3 if model.context.kind=='multi_view_kernel'else 1,
        parameter_matching='Counts reported; extra-view model has additional active maps/membership parameters',
        dense_pair_complexity='O(B H N² S V); no FMM implementation or speed claim')
    return result

@torch.no_grad()
def kernel_geometry_diagnostics(model_or_encoder,images,indices=None,valid=None,max_samples=2,
                                far_block_ranks=(8,16,32),separation_factor=1.25):
    encoder=getattr(model_or_encoder,'context',model_or_encoder)
    if not isinstance(encoder,OLD.KernelVisionEncoder):return {'status':'NOT_APPLICABLE','reason':'ViT encoder'}
    device=next(encoder.parameters()).device;images=images[:max_samples].to(device)
    if indices is not None:indices=indices[:max_samples].to(device)
    if valid is not None:valid=valid[:max_samples].to(device)
    prior=encoder.training;encoder.eval();layers=[]
    try:
        x,indices,valid,grid=encoder.embed(images,indices,valid)
        for index,block in enumerate(encoder.blocks):
            mixer=block.mixer;state=mixer.kernel_state(block.norm1(x),indices,valid,grid)
            q,k,w=state['query_coordinates'],state['source_coordinates'],state['weights'];rowmask=valid[:,None].expand(-1,mixer.heads,-1)
            entropy=-(w*w.clamp_min(torch.finfo(w.dtype).tiny).log()).sum(-1)[rowmask];participation=1/w.square().sum(-1)[rowmask]
            ellp=mixer.log_bandwidths.clamp(math.log(.03),math.log(2.)).exp()
            ellc=(mixer.log_content_bandwidths if hasattr(mixer,'log_content_bandwidths')else mixer.log_bandwidths).clamp(math.log(.03),math.log(2.)).exp()
            row={'block':index,'variant':encoder.kind,'coordinate_dimension':mixer.coordinate_dim,
                'geometry':state.get('geometry','positions×bounded six-dimensional semantic embedding'),
                'bandwidths':ellp.cpu().tolist(),'content_bandwidths':ellc.cpu().tolist(),'mixture_weights':mixer.mixture_logits.softmax(-1).cpu().tolist(),
                'kernel_entropy_mean':float(entropy.mean()),'effective_neighbors_exp_entropy_mean':float(entropy.exp().mean()),
                'effective_neighbors_participation_mean':float(participation.mean()),'query_source_displacement_mean':float((q-k).norm(dim=-1)[rowmask].mean()),
                'coordinate_spread_mean_std':float(q.transpose(1,2)[valid].std(dim=0,unbiased=False).mean()),
                'content_coordinate_rms':float(q[...,2:].square().mean().sqrt()),'views':getattr(mixer,'views',1),'far_blocks':[]}
            if encoder.kind=='sphere_kernel':row.update(query_unit_norm_max_error=float((q[...,2:].norm(dim=-1)[rowmask]-1).abs().max()),
                source_unit_norm_max_error=float((k[...,2:].norm(dim=-1)[rowmask]-1).abs().max()),zero_fallback_count=sum(int(mask[rowmask].sum())for mask in state['zero_fallback_masks']))
            if encoder.kind=='multi_view_kernel':row.update(source_membership_mean=state['source_memberships'].permute(0,3,1,2)[valid].mean((0,1)).cpu().tolist(),
                receiver_membership_mean=state['receiver_memberships'].permute(0,3,1,2)[valid].mean((0,1)).cpu().tolist(),
                view_coordinate_geometry='Three bounded6Dsemantic embeddings; firstview shares baseline initialization. No manifold chart/injectivity claim.')
            layers.append(row);x=block(x,indices,valid,grid)
        return {'status':'MEASURED','variant':encoder.kind,'grid':list(grid),'samples':len(images),'layers':layers,'dense_only':True,'diagnostic_device':str(device),
            'interpretation':'Descriptive learned embeddings and normalized kernels. Sphere/product domains are explicit; image-manifold isomorphism, chart structure and FMM speedup are not established.'}
    finally:encoder.train(prior)
